Wei J, Wang X, Schuurmans D, et al. Chain-of-thought prompting elicits reasoning in large language models[J]. Advances in Neural Information Processing Systems, 2022, 35: 24824-24837.

In [ ]:
from dotenv import load_dotenv
load_dotenv('../.env')

import os
from openai import OpenAI
from functools import lru_cache

class BaseLLM:
    @lru_cache(maxsize=1024)
    def chat(self, text):
        return self._chat(text)
    def _chat(self, text):
        raise NotImplementedError
        
class OpenAILLM(BaseLLM):
    def __init__(self, model_name):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=os.getenv("OPENAI_API_BASE")
        )
        self.model_name = model_name
        self.conversation_history = []
        
    def convert_text_to_prompt(self,instr,target):
        return instr.format(target)

    def __call__(self,text):
        return self.chat(text)
        
    def chat(self, text, temperature=0, messages=[], stops=None):
        return self._chat(text, temperature, messages, stops)
        
    def _chat(self, text, temperature, messages=[], stops=None):
        if not messages:
            messages = [{"role": "user", "content": text}]
        print("开始请求模型%s"%(self.model_name))
        
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=messages,
            stream=False,
            stop=stops,
            temperature= temperature,
        )
        
        return response.choices[0].message.content
        
    def history_chat(self,text, message=[], stops=None):
        self.conversation_history.append({"role": "user", "content": text})

        # 创建请求
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self.conversation_history,  # 传递完整的对话历史
            stream=False,
        )

        # 将模型的回答添加到对话历史中
        answer = response.choices[0].message.content
        self.conversation_history.append({"role": "assistant", "content": answer})
        return answer
        
gpt_4 = OpenAILLM("gpt-4")

gpt_3_5_turbo = OpenAILLM("gpt-3.5-turbo")

In [ ]:
#Bad Case

gpt_3_5_turbo.chat("""
Q:小明每天早上花费20分钟时间走到学校，如果小明家距离学校225335厘米，那么他每一分钟走多少米?
A:
""")

In [ ]:
#Good Case
#模型模仿例子的思维过程得到正确答案
gpt_3_5_turbo.chat("""
Q:两辆汽车从相距500千米的两城同时出发，相向而行。一辆摩托车以每小时80千米的速度在两辆汽车之间不断往返联络。已知这两辆汽车的速度分别是每小时40千米和60千米，求两汽车相遇时，摩托车共行驶了多少千米?
A:我们可以使用相遇前的时间来计算摩托车的行驶距离。两辆汽车的相对速度是40千米/小时+60千米/小时 =100千米/小时 根据题目我们可以得知，两辆汽车从相距500千米的两个城市同时出发，因此它们相遇的时间是 500千米/100千米/小时 =5小时 摩托车以每小时80千米的速度在两辆汽车之间往返，因此它每小时行驶的距离是80千米。所以，摩托车共行驶了5小时乘以80千米/小时 =400千米
Q:小明每天早上花费20分钟时间走到学校，如果小明家距离学校225335厘米，那么他每一分钟走多少米?
A:
""")

In [ ]:
#Bad Case
#模型无法得到正确答案
#正确答案是44岁
gpt_3_5_turbo.chat("""
Q:我的妈妈在29岁时生下的我。今年我14岁了，我的妹妹是我年龄的一半。当我妈妈已经80岁时，我的妹妹几岁？
A:
""")

In [ ]:
#Good Case
#模型通过学习提示中的自我一致性信息得到正确答案
#论文中是在CoT提示中进行采样，以生成多个推理路径，得到多个推理结果，再选择最一致的结果
Prompt = """
Q：林中有15棵树。林业工人今天将在林中种树。完成后，将有21棵树。林业工人今天种了多少棵树？
A：我们从15棵树开始。后来我们有21棵树。差异必须是他们种树的数量。因此，他们必须种了21-15 = 6棵树。答案是6。

Q：停车场有3辆汽车，又来了2辆汽车，停车场有多少辆汽车？
A：停车场已经有3辆汽车。又来了2辆。现在有3 + 2 = 5辆汽车。答案是5。

Q：Leah有32块巧克力，她的姐姐有42块。如果他们吃了35块，他们总共还剩多少块？
A：Leah有32块巧克力，Leah的姐姐有42块。这意味着最初有32 + 42 = 74块巧克力。已经吃了35块。因此，他们总共还剩74-35 = 39块巧克力。答案是39。

Q：Jason有20个棒棒糖。他给Denny一些棒棒糖。现在Jason只有12个棒棒糖。Jason给Denny多少棒棒糖？
A：Jason有20个棒棒糖。因为他现在只有12个，所以他必须把剩下的给Denny。他给Denny的棒棒糖数量必须是20-12 = 8个棒棒糖。答案是8。

Q：Shawn有五个玩具。圣诞节，他从他的父母那里得到了两个玩具。他现在有多少个玩具？
A：他有5个玩具。他从妈妈那里得到了2个，所以在那之后他有5 + 2 = 7个玩具。然后他从爸爸那里得到了2个，所以总共他有7 + 2 = 9个玩具。答案是9。

Q：服务器房间里有9台计算机。从周一到周四，每天都会安装5台计算机。现在服务器房间里有多少台计算机？
A：从周一到周四有4天。每天都添加了5台计算机。这意味着总共添加了4 * 5 =
20台计算机。一开始有9台计算机，所以现在有9 + 20 = 29台计算机。答案是29。

Q：Michael有58个高尔夫球。星期二，他丢失了23个高尔夫球。星期三，他又丢失了2个。星期三结束时他还剩多少个高尔夫球？
A：Michael最初有58个球。星期二他丢失了23个，所以在那之后他有58-23 = 35个球。星期三他又丢失了2个，所以现在他有35-2 = 33个球。答案是33。

Q：Olivia有23美元。她用每个3美元的价格买了五个百吉饼。她还剩多少钱？
A：她用每个3美元的价格买了5个百吉饼。这意味着她花了15美元。她还剩8美元。

Q：我的妈妈在29岁时生下的我。今年我14岁了，我的妹妹是我年龄的一半。当我妈妈已经80岁时，我的妹妹几岁？
A：
"""
print(gpt_4.chat(Prompt,temperature=0.1))
print(gpt_4.chat(Prompt,temperature=0.7))
print(gpt_4.chat(Prompt,temperature=0.3))

In [ ]:
print(gpt_3_5_turbo.chat("你知道24点吗，请算出4 5 6 10的24点"))

In [ ]:
print(gpt_3_5_turbo.chat("你知道24点吗，请一步一步算出4 5 6 10的24点"))

In [ ]:
 !pip install tree-of-thoughts-llm

In [ ]:
def solve(args, task, idx, to_print=True):
    # 声明gpt为全局变量，并根据参数配置gpt函数。
    global gpt
    gpt = partial(gpt, model=args.backend, temperature=args.temperature)
    print(gpt)  # 打印当前配置的gpt函数，以便了解其设置。

    # 根据给定的索引从任务中获取输入数据。
    x = task.get_input(idx)
    
    # 初始化输出候选列表，开始时只包含一个空字符串。
    ys = ['']
    
    # 用于记录每步的信息。
    infos = []
    
    # 对于任务定义的每一步骤进行循环。
    for step in range(task.steps):
        # 根据参数args.method_generate选择生成方法。
        if args.method_generate == 'sample':
            # 如果方法为'sample'，则对每个候选y生成新的候选列表。
            new_ys = [get_samples(task, x, y, args.n_generate_sample, prompt_sample=args.prompt_sample, stop=task.stops[step]) for y in ys]
        elif args.method_generate == 'propose':
            # 如果方法为'propose'，则获取提案。
            new_ys = [get_proposals(task, x, y) for y in ys]
        # 将所有新候选列表合并为一个列表。
        new_ys = list(itertools.chain(*new_ys))

        # 准备评估新候选。
        ids = list(range(len(new_ys)))
        if args.method_evaluate == 'vote':
            # 如果评估方法为'vote'，则通过投票获取每个新候选的评分。
            values = get_votes(task, x, new_ys, args.n_evaluate_sample)
        elif args.method_evaluate == 'value':
            # 如果评估方法为'value'，则直接获取每个新候选的值评估。
            values = get_values(task, x, new_ys, args.n_evaluate_sample)

        # 根据评分选择新的候选。
        if args.method_select == 'sample':
            # 如果选择方法为'sample'，通过概率抽样选择候选。
            ps = np.array(values) / sum(values)
            select_ids = np.random.choice(ids, size=args.n_select_sample, p=ps).tolist()
        elif args.method_select == 'greedy':
            # 如果选择方法为'greedy'，则贪心选择最高评分的候选。
            select_ids = sorted(ids, key=lambda x: values[x], reverse=True)[:args.n_select_sample]
        select_new_ys = [new_ys[select_id] for select_id in select_ids]

        # 如果需要，打印每步的详细信息。
        if to_print:
            sorted_new_ys, sorted_values = zip(*sorted(zip(new_ys, values), key=lambda x: x[1], reverse=True))
            print(f'-- new_ys --: {sorted_new_ys}\n-- sol values --: {sorted_values}\n-- choices --: {select_new_ys}\n')
        
        # 记录这一步的信息。
        infos.append({'step': step, 'x': x, 'ys': ys, 'new_ys': new_ys, 'values': values, 'select_new_ys': select_new_ys})
        # 更新当前输出候选为选择的新候选。
        ys = select_new_ys
    
    # 打印最终选择的候选解决方案。
    if to_print:
        print(ys)
    # 返回最终的候选解决方案及过程信息。
    return ys, {'steps': infos}

In [ ]:
import argparse
from tot.methods.bfs import solve
from tot.tasks.game24 import Game24Task

args = argparse.Namespace(backend='gpt-4', temperature=0.7, task='game24', naive_run=False, prompt_sample=None, method_generate='propose', method_evaluate='value', method_select='greedy', n_generate_sample=1, n_evaluate_sample=3, n_select_sample=5)

task = Game24Task()
ys, infos = solve(args, task, 900)
print(ys[0])

https://github.com/princeton-nlp/tree-of-thought-llm

https://github.com/spcl/graph-of-thoughts

In [ ]:
def solve(self):
    """使用AoT提示和DFS搜索算法解决问题"""
    try:
        # 运行DFS
        self.dfs(self.initial_prompt, 1)  # 基于初始提示执行深度优先搜索，1可能表示从某个初始深度开始搜索

        # 检查是否生成了任何思路
        if not self.output:  # 如果没有输出（即没有找到有效的思路）
            logger.error("在DFS期间没有生成有效的思路")  # 记录错误日志
            return None  # 没有思路则返回None

        # 找到最佳思路及其值
        best_state, best_value = max(self.output, key=lambda x: x[1])  # 从输出中找到最佳思路及其评分值

        # 缓存最佳思路
        self.thought_cache["accepted"][best_state] = best_value  # 将最佳思路及其评分值存入缓存

        # 根据最佳思路生成最终解决方案
        solution = self.model.generate_solution(self.initial_prompt, best_state)  # 使用模型根据最佳思路生成解决方案

        # 显示并返回解决方案
        print(f"解决方案是 {solution}")  # 打印解决方案

        # 将缓存写入JSON文件
        # 如果希望覆盖文件，请将模式改回'w'
        with open("./thought_cache.json", "a") as json_file:  # 以追加模式打开或创建缓存文件
            json.dump(self.thought_cache, json_file)  # 将缓存数据写入文件

        return solution if solution else best_state  # 如果有解决方案则返回解决方案，否则返回最佳状态

    except Exception as error:
        logger.error(f"在tot_dfs中发生错误: {error}")  # 记录发生的错误

        # 即使发生错误也将缓存写入JSON文件
        # 如果希望覆盖文件，请将模式改回'w'
        with open("./thought_cache_error.json", "a") as json_file:  # 以追加模式打开或创建错误缓存文件
            json.dump(self.thought_cache, json_file)  # 将缓存数据写入文件

        raise error  # 抛出错误

def dfs(self, state, step):
    """深度优先搜索算法"""
    if step > self.max_steps:
        # 超过最大步数限制时的处理
        if state in self.thought_cache["accepted"]:
            # 如果当前状态已经在“接受”的缓存中，则直接获取其值
            value = self.thought_cache["accepted"][state]
        elif state in self.thought_cache["pruned"]:
            # 如果当前状态在“剪枝”的缓存中，则直接返回不进行处理
            return
        else:
            # 否则，评估当前的思考（状态），并将其结果缓存
            thought, value = self.evaluate_thought(state)
            self.thought_cache["accepted"][state] = value

        # 记录当前状态及其评估值
        self.output.append((state, value))
        return

    # 在生成和过滤思考之前检查缓存
    if state in self.thought_cache["accepted"]:
        thoughts = [state]
    elif state in self.thought_cache["pruned"]:
        return
    else:
        # 生成和过滤当前状态的后续思考
        thoughts = self.generate_and_filter_thoughts(state)

    for next_state in thoughts:
        # 获取下一个状态的评估值，如果未评估则默认为0
        state_value = self.evaluated_thoughts.get(next_state, 0)
        print("Entering DFS with state: ", state, " and step: ", step)

        # 如果下一个状态的评估值低于阈值，则将其加入“剪枝”缓存并继续循环
        if state_value <= self.value_threshold:
            self.thought_cache["pruned"][next_state] = state_value
            continue

        # 继续进行深度优先搜索
        child = (
            (state, next_state) if isinstance(state, str) else (*state, next_state)
        )
        self.dfs(child, step + 1)

        # 回溯
        # 找出当前输出中的最大值
        best_value = max([value for _, value in self.output])

        # 如果最佳值低于回溯阈值，则从输出中移除最后一个元素并继续
        if best_value < self.backtracking_threshold:
            self.output.pop()
            continue

In [ ]:
# !pip install aot-x

In [ ]:
from aot.main import AoT
import os

task = """

Use numbers and basic arithmetic operations (+ - * /) to obtain 24. When
considering the next steps, do not choose operations that will result in a
negative or fractional number. In order to help with the calculations, the
numbers in the parenthesis represent the numbers that are left after the
operations and they are in descending order.
Another thing we do is when there are only two numbers left in the parenthesis, we
check whether we can arrive at 24 only by using basic arithmetic operations
(+ - * /). Some examples regarding this idea:
(21 2) no
since 21 + 2 = 23, 21 - 2 = 19, 21 * 2 = 42, 21 / 2 = 10.5, none of which is equal
to 24.
(30 6) 30 - 6 = 24 yes
(8 3) 8 * 3 = 24 yes
(12 8) no
(48 2) 48 / 2 = 24 yes
Most importantly, do not give up, all the numbers that will be given has indeed a
solution.

5 6 10 4
"""
os.environ['OPENAI_API_MODEL'] = 'gpt-4'


dfs = AoT(
    num_thoughts=5,
    max_steps=10, 
    value_threshold=1,
    initial_prompt=task,
)

result = dfs.solve()
print(result)

https://github.com/kyegomez/Algorithm-Of-Thoughts